In [ ]:
# CELL 1 — fast, reproducible Friday DDoS subset download (automatic)
%pip -q install pandas numpy scikit-learn matplotlib seaborn requests
from pathlib import Path
import requests, pandas as pd
BASE = Path('/content/project13_cicids2017'); RAW=BASE/'raw'; OUT=BASE/'outputs'; FIG=BASE/'figures'
for folder in (RAW, OUT, FIG): folder.mkdir(parents=True,exist_ok=True)
SOURCE_URL='https://www.unb.ca/cic/datasets/ids-2017.html'
SOURCE_MIRROR='https://huggingface.co/datasets/c01dsnap/CIC-IDS2017/resolve/main/Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv'
MAX_SOURCE_CSV_FILES=1
filename='Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv'
destination=RAW/filename

def check_csv(path):
    if not path.exists() or path.stat().st_size < 1000: return False
    try:
        columns=[str(c).strip() for c in pd.read_csv(path,nrows=10,encoding='utf-8-sig').columns]
        return 'Label' in columns and len(columns)>=40
    except Exception: return False

if not check_csv(destination):
    print('Downloading one CICIDS2017 Friday DDoS CSV (not the full dataset)...')
    temporary=RAW/(filename+'.part')
    try:
        with requests.get(SOURCE_MIRROR,stream=True,timeout=(20,60),headers={'User-Agent':'Colab-Research-Notebook'}) as response:
            response.raise_for_status()
            total=0
            with open(temporary,'wb') as output:
                for chunk in response.iter_content(chunk_size=1024*1024):
                    if chunk:
                        output.write(chunk); total+=len(chunk)
                        if total%(10*1024*1024)<len(chunk): print(f'Downloaded {total/1e6:.1f} MB')
        if not check_csv(temporary): raise ValueError('Downloaded file failed CSV schema validation')
        temporary.replace(destination)
    except Exception as error:
        temporary.unlink(missing_ok=True)
        raise RuntimeError('Subset download failed; check the mirror or supply an authorized CSV. Details: '+str(error)) from error
sources=[destination]
print('CELL 1 COMPLETE —',len(sources),'file; size:',round(destination.stat().st_size/1e6,1),'MB')
print('Scope: Friday afternoon BENIGN/DDoS subset only; historical research data, not live traffic.')


In [ ]:
# CELL 2 — bounded randomized flow sample; validate labels and schema
import pandas as pd, numpy as np, json
RNG_SEED=42
CHUNKSIZE=40000
ROW_SAMPLE_PROB=0.07  # reproducible Bernoulli subsample, not full population
MAX_SAMPLE_ROWS=220000
rng=np.random.default_rng(RNG_SEED)
parts=[]; inspection=[]; sampled_total=0
for source in sources:
    source_rows=0; kept=0
    for chunk in pd.read_csv(source, chunksize=CHUNKSIZE, low_memory=False, on_bad_lines='skip', encoding='utf-8-sig'):
        chunk.columns=[str(c).strip() for c in chunk.columns]
        if 'Label' not in chunk.columns: raise ValueError(f'Missing Label column in {source.name}')
        source_rows+=len(chunk)
        mask=rng.random(len(chunk))<ROW_SAMPLE_PROB
        sub=chunk.loc[mask].copy()
        if not sub.empty:
            sub['source_file']=source.name
            parts.append(sub)
            kept+=len(sub);sampled_total+=len(sub)
        if sampled_total>=MAX_SAMPLE_ROWS: break
    inspection.append({'source_file':source.name,'rows_read':source_rows,'sampled_rows':kept})
    if sampled_total>=MAX_SAMPLE_ROWS: break
assert parts,'Sampling found no data'
df=pd.concat(parts,ignore_index=True,sort=False)
assert 'Label' in df.columns
# If a Kaggle variant includes duplicated names, keep first and explicitly record occurrence.
duplicated_names=df.columns.duplicated().sum()
if duplicated_names: df=df.loc[:,~df.columns.duplicated()].copy()
df['Label']=df['Label'].astype('string').fillna('UNKNOWN').str.strip()
df['is_attack']=~df['Label'].str.upper().eq('BENIGN')
assert df['is_attack'].nunique()==2,'Need both BENIGN and attack labels to fit baseline; try larger sample.'
df.to_csv(OUT/'sampled_flow_labels.csv',index=False,columns=['source_file','Label','is_attack'])
pd.DataFrame(inspection).to_csv(OUT/'source_sampling_audit.csv',index=False)
print('Sampled rows:',len(df),'attack rows:',int(df['is_attack'].sum()),'benign:',int((~df['is_attack']).sum()))
print('Source sampling is Bernoulli 7% with cap 220k, subject to available files; not full-dataset statistics.')
print('Observed label categories:',df['Label'].value_counts().to_dict())

In [ ]:
# CELL 3 — data-quality audit and SQLite segmentation
import sqlite3
null_counts=df.isna().sum()
num_cols=[c for c in df.columns if c not in ['Label','source_file','is_attack']]
missing=int(df.isna().sum().sum())
audit=pd.DataFrame([{'sampled_rows':len(df),'files_used':df['source_file'].nunique(),
 'sampled_columns':len(df.columns),'missing_cells':missing,
 'duplicate_sample_rows':int(df.drop(columns=['source_file']).duplicated().sum()),
 'duplicated_column_names_before_fix':int(duplicated_names),
 'benign_records':int((~df['is_attack']).sum()),'attack_records':int(df['is_attack'].sum())}])
audit.to_csv(OUT/'data_quality_audit.csv',index=False)
labels=df.groupby('Label',dropna=False).size().reset_index(name='records').sort_values('records',ascending=False)
labels['sample_pct']=(labels.records/len(df)*100).round(3)
labels.to_csv(OUT/'attack_category_breakdown.csv',index=False)
with sqlite3.connect(':memory:') as con:
    dsql=df[['source_file','Label','is_attack']].copy()
    dsql['is_attack']=dsql['is_attack'].astype(int)
    dsql.to_sql('flows',con,index=False,if_exists='replace',chunksize=5000)
    source_by_label=pd.read_sql_query("SELECT source_file,Label,COUNT(*) AS records,SUM(is_attack) AS attack_records FROM flows GROUP BY source_file,Label ORDER BY source_file,records DESC",con)
    source_by_label.to_csv(OUT/'sql_source_label_breakdown.csv',index=False)
    check=pd.read_sql_query('SELECT COUNT(*) AS n,SUM(is_attack) AS a FROM flows',con).iloc[0]
    assert int(check['n'])==len(df) and int(check['a'])==int(df.is_attack.sum())
print('SQLite reconciliation: PASS; sampled columns:',len(df.columns))
print(audit.to_string(index=False))

In [ ]:
# CELL 4 — defensive binary baseline; random split with explicit leakage caveat
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score, balanced_accuracy_score
# Random split supports a basic classroom benchmark, NOT temporal/generalization evidence.
DROP_COLS={'Label','source_file','is_attack','Flow ID','Source IP','Destination IP','Src IP','Dst IP','Timestamp','SimillarHTTP','Unnamed: 0'}
feat=[c for c in df.columns if c not in DROP_COLS and not c.lower().startswith('unnamed')]
X=df[feat].apply(pd.to_numeric,errors='coerce').replace([np.inf,-np.inf],np.nan)
feat=[c for c in X.columns if X[c].notna().any()]
X=X[feat]
y=df['is_attack'].astype(int)
train_idx,test_idx=train_test_split(np.arange(len(df)),test_size=0.25,stratify=y,random_state=RNG_SEED)
imputer=SimpleImputer(strategy='median',add_indicator=False,keep_empty_features=False)
X_train=imputer.fit_transform(X.iloc[train_idx]); X_test=imputer.transform(X.iloc[test_idx])
model=RandomForestClassifier(n_estimators=70,max_depth=15,min_samples_leaf=3,n_jobs=-1,random_state=RNG_SEED,class_weight='balanced_subsample')
model.fit(X_train,y.iloc[train_idx]); pred=model.predict(X_test)
assert len(pred)==len(test_idx)
print('Train/test sample:',len(train_idx),len(test_idx),'features:',len(feat))

In [ ]:
# CELL 5 — held-out diagnostics and category-level misses
truth=y.iloc[test_idx].to_numpy(); tn,fp,fn,tp=confusion_matrix(truth,pred,labels=[0,1]).ravel()
metrics={'dataset':'CICIDS2017','sampling_probability':ROW_SAMPLE_PROB,'sample_cap':MAX_SAMPLE_ROWS,
 'sample_rows':len(df),'train_rows':len(train_idx),'test_rows':len(test_idx),'features_used':len(feat),
 'random_seed':RNG_SEED,'split':'stratified random 75/25 split on sampled flows (not temporal or cross-day)',
 'accuracy':float(accuracy_score(truth,pred)),
 'balanced_accuracy':float(balanced_accuracy_score(truth,pred)),
 'attack_precision':float(precision_score(truth,pred,zero_division=0)),
 'attack_recall':float(recall_score(truth,pred,zero_division=0)),
 'attack_f1':float(f1_score(truth,pred,zero_division=0)),
 'TN':int(tn),'FP':int(fp),'FN':int(fn),'TP':int(tp),
 'false_positive_rate':float(fp/(fp+tn)) if fp+tn else None,
 'miss_rate':float(fn/(fn+tp)) if fn+tp else None,
 'not_live_soc':True,'limitation':'Historical benchmark; random split can overstate transfer to unseen days or live traffic.'}
(OUT/'holdout_model_metrics.json').write_text(json.dumps(metrics,indent=2)+'\n')
perf=pd.DataFrame({'Label':df.iloc[test_idx]['Label'].to_numpy(),'actual_attack':truth,'predicted_attack':pred})
per_label=perf.groupby('Label').agg(records=('actual_attack','size'),attacks=('actual_attack','sum'),flagged=('predicted_attack','sum')).reset_index()
per_label['missed_attacks']=perf.assign(missed=((perf.actual_attack==1)&(perf.predicted_attack==0)).astype(int)).groupby('Label')['missed'].sum().reindex(per_label.Label).to_numpy()
per_label.to_csv(OUT/'category_detection_performance.csv',index=False)
feature_importance=pd.DataFrame({'feature':feat,'importance':model.feature_importances_}).sort_values('importance',ascending=False)
feature_importance.to_csv(OUT/'feature_importance.csv',index=False)
assert tn+fp+fn+tp==len(test_idx)
print(json.dumps(metrics,indent=2))

In [ ]:
# CELL 6 — verified static PNG executive dashboard
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
fig=plt.figure(figsize=(14,10),facecolor='#0b1725')
ax1=fig.add_axes([.07,.48,.40,.36],facecolor='#10263b')
ax2=fig.add_axes([.56,.48,.38,.36],facecolor='#10263b')
ax3=fig.add_axes([.07,.12,.87,.22],facecolor='#10263b')
fig.text(.07,.94,'CICIDS2017 | DEFENSIVE BENCHMARK ANALYTICS',fontsize=19,color='white',weight='bold')
fig.text(.07,.905,'Historical sampled research flows • not live threats • random-split evaluation caveat',fontsize=11,color='#bed0dc')
for i,(v,label) in enumerate([(metrics['attack_precision'],'ATTACK PRECISION'),(metrics['attack_recall'],'ATTACK RECALL'),(metrics['false_positive_rate'],'FALSE-POSITIVE RATE'),(metrics['miss_rate'],'ATTACK MISS RATE')]):
    x=.07+i*.222
    fig.text(x,.865,f'{v:.1%}',fontsize=24,color='#5be0c4',weight='bold')
    fig.text(x,.837,label,fontsize=9,color='#c2d6e3')
b=labels.head(8).iloc[::-1];ax1.barh(b.Label,b.records,color='#33a3c7')
ax1.set_title('Observed sample composition (top labels)',color='white',pad=14)
cat=per_label.loc[per_label.attacks>0].sort_values('missed_attacks',ascending=False).head(8).iloc[::-1]
ax2.barh(cat.Label,cat.missed_attacks,color='#ffa66c');ax2.set_title('Missed attacks by dataset label',color='white',pad=14)
ax3.bar(['True negative','False positive','False negative','True positive'],[tn,fp,fn,tp],color=['#45c6ac','#ffaf6b','#f07e73','#338ac2'])
ax3.set_title('Held-out confusion matrix counts',color='white',pad=12)
for ax in (ax1,ax2,ax3):
    ax.tick_params(colors='#dbe7ef',labelsize=9)
    for spine in ax.spines.values(): spine.set_color('#36536b')
    ax.grid(axis='x' if ax in (ax1,ax2) else 'y',color='#31506a',alpha=.35)
fig.text(.07,.036,f"Sample {len(df):,} rows | Train {len(train_idx):,} | Test {len(test_idx):,} | files {len(inspection)} | seed {RNG_SEED}",color='#bed0dc',fontsize=10)
fig.savefig(FIG/'soc_executive_dashboard.png',dpi=160,facecolor=fig.get_facecolor(),bbox_inches='tight')
plt.show();plt.close(fig)
print('Saved:',FIG/'soc_executive_dashboard.png')

In [ ]:
# CELL 7 — recommendations, documentation and reproducibility notice
rec=[
(1,'Avoid live-SOC interpretations','CICIDS2017 is a historical research benchmark.','Label every artifact as retrospective; do not claim current threats.','No production claims.'),
(2,'Treat random-split scores as a baseline','Rows were randomly stratified after subsampling; related flows may span train/test.','Run cross-day or group-held-out follow-up where feasible.','Publish temporal/group evaluation.'),
(3,'Review missed attacks',f"Held-out false negatives: {metrics['FN']:,}; recall {metrics['attack_recall']:.1%}.",'Inspect top missed attack labels in category_detection_performance.csv.','Track missed counts and recall by label.'),
(4,'Control false alerts',f"Held-out false positives: {metrics['FP']:,}; FPR {metrics['false_positive_rate']:.1%}.",'Review benign false flags before operational escalation plans.','Track false-positive rate alongside recall.'),
(5,'Validate ingestion and schema',f"{len(sources)} source CSVs selected; {len(df):,} sampled rows.",'Track file coverage, schema, missingness, infinity and row filtering.','Reconcile sampled and SQL totals.'),
(6,'Make class imbalance visible',f"Observed BENIGN records: {int((~df.is_attack).sum()):,}; attack records: {int(df.is_attack.sum()):,}.",'Preserve per-label and per-file distributions; avoid blanket prevalence claims.','Publish category supports.'),
(7,'Require contemporary authorized validation','Historical academic flows do not establish deployed accuracy.','Test under privacy-approved, modern, distinct traffic conditions.','No deployment decision based on this notebook alone.')]
recs=pd.DataFrame(rec,columns=['priority','recommendation','evidence','action','success_measure'])
recs.to_csv(OUT/'recommendations.csv',index=False)
source_notice="""# Data source and responsible use
Official source: https://www.unb.ca/cic/datasets/ids-2017.html
Automatic mirror: https://www.kaggle.com/datasets/cicdataset/cicids2017
Source CSVs are not included in the project package. Respect upstream terms and citation requirements.
The data represent a historical academic benchmark, not live enterprise incident reporting.
This analysis is defensive and educational.
"""
(BASE/'LICENSE_DATA_NOTICE.md').write_text(source_notice)
(BASE/'requirements.txt').write_text('kagglehub\npandas\nnumpy\nscikit-learn\nmatplotlib\nseaborn\n')
(BASE/'.gitignore').write_text('raw/\n*.zip\n__pycache__/\n.ipynb_checkpoints/\n')
readme=f"""# CICIDS2017 Intrusion Detection & SOC Analytics — Project 13

## Executive summary
Defensive analysis of sampled **historical CICIDS2017 academic network flows** using Python, SQLite, a RandomForest baseline, a static SOC-style dashboard and seven evidence-based recommendations. **Not a live SOC monitoring system.**

## Sources
- Official research source: https://www.unb.ca/cic/datasets/ids-2017.html
- Automatic mirror: https://www.kaggle.com/datasets/cicdataset/cicids2017
- Input: working-hours labeled CSV flow files; original source records not redistributed.

## Sampling and scope
- Sampled **{len(df):,}** rows from **{len(inspection)}** CSV files with seed **{RNG_SEED}**, Bernoulli probability **{ROW_SAMPLE_PROB:.0%}**, cap **{MAX_SAMPLE_ROWS:,}** records.
- Observed labels and rates describe **this sample**, not population prevalence, live traffic or all CICIDS2017 files.
- Some datasets may contain timestamps but this project intentionally does **not** claim timeline analysis.
- Data loading skips malformed CSV lines and numeric conversion errors are imputed for modeling; see audit outputs.

## Baseline method and caution
- 75/25 stratified **random split after sampling**, train-only median imputation, RandomForestClassifier. This may overstate transfer to future days and related flows; **not temporal holdout, not production evaluation**.
- Held-out training rows: **{len(train_idx):,}**, testing rows: **{len(test_idx):,}**.
- Precision: **{metrics['attack_precision']:.4f}**; recall: **{metrics['attack_recall']:.4f}**; F1: **{metrics['attack_f1']:.4f}**.
- TN **{tn:,}**, FP **{fp:,}**, FN **{fn:,}**, TP **{tp:,}**.

## Visualization
![Historical SOC-style analytical dashboard](figures/soc_executive_dashboard.png)

## Seven recommendations
"""
for _,r in recs.iterrows():readme+=f"{r['priority']}. **{r['recommendation']}** — Evidence: {r['evidence']} Action: {r['action']} KPI: {r['success_measure']}\n"
readme+="""
## Reproduce on Google Colab
1. Open `Project_13_CICIDS2017_SOC_Analytics_AutoDownload.ipynb`.
2. Run cells 1–8 in order; Cell 1 tries a public Kaggle download and offers upload fallback.
3. Review the source file list and sample counts before drawing conclusions.
4. Cell 8 writes a project ZIP and downloadable CSV/PNG/docs, excluding upstream source flow records.
5. Upload the notebook and extracted ZIP contents **with folders intact** to GitHub.

## Generated files
- `figures/soc_executive_dashboard.png`
- `outputs/*.csv`, `outputs/holdout_model_metrics.json`
- `README.md`, `requirements.txt`, `LICENSE_DATA_NOTICE.md`, `FILE_MANIFEST.csv`, `PROJECT_STATUS.md`

## Limitations
Historical experimental traffic; sampled data; variable mirror availability; no claim of operational threat prevalence or real-time alerts; random-row split has potential related-flow leakage. Validate on recent authorized data before use.
"""
(BASE/'README.md').write_text(readme)
(BASE/'PROJECT_STATUS.md').write_text('STATUS: LOCALLY GENERATED — GitHub upload and live verification are pending. Do not mark FINAL PASS until checked.\n')
print('Saved 7 evidence-backed recommendations and project documentation')

In [ ]:
# CELL 8 — verify package, manifest, ZIP, and Colab download
import hashlib, zipfile
required=[BASE/'README.md',BASE/'PROJECT_STATUS.md',BASE/'requirements.txt',BASE/'LICENSE_DATA_NOTICE.md',FIG/'soc_executive_dashboard.png',OUT/'recommendations.csv',OUT/'holdout_model_metrics.json']
for f in required: assert f.exists() and f.stat().st_size>0,f'Missing generated file: {f}'
assert len(pd.read_csv(OUT/'recommendations.csv'))==7
assert sum(metrics[k] for k in ('TN','FP','FN','TP'))==metrics['test_rows']
# Manifest deliberately excludes itself to avoid self-referential checksums.
files_to_package=sorted(p for p in BASE.rglob('*') if p.is_file() and RAW not in p.parents and p.name not in {'FILE_MANIFEST.csv'})
manifest=pd.DataFrame([{'file':str(p.relative_to(BASE)), 'bytes':p.stat().st_size,
 'sha256':hashlib.sha256(p.read_bytes()).hexdigest()} for p in files_to_package])
manifest.to_csv(BASE/'FILE_MANIFEST.csv',index=False)
zip_path=Path('/content/Project_13_CICIDS2017_SOC_Analytics_GitHub.zip')
with zipfile.ZipFile(zip_path,'w',compression=zipfile.ZIP_DEFLATED) as z:
 for f in [*files_to_package,BASE/'FILE_MANIFEST.csv']:z.write(f,arcname=str(f.relative_to(BASE)))
with zipfile.ZipFile(zip_path) as z:
 assert z.testzip() is None
print('VALIDATION PASS: 7 recommendations, correct confusion-matrix totals, checksummed files, readable ZIP')
print('ZIP:',zip_path,'bytes:',zip_path.stat().st_size)
from google.colab import files
files.download(str(zip_path))